# QC Anomaly Detection - Interactive Analysis Notebook

This notebook demonstrates the complete workflow for QC anomaly detection:
1. Load data
2. Define filters
3. Generate filtered dataset
4. Run anomaly detection
5. Visualize results

**Note**: This is a simulation of the final system. It is used for testing purpose during build

## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import sys

## 1. test Data loading scripts/functions


In [ ]:
from src.data_validator import validate_structure, validate_lcs_data

# Ensure the project root is on the path so src.* imports work
sys.path.insert(0, str(Path(".").resolve()))

data_filepath = r"data\raw\QC_Sample_Data.csv"  # UPDATE IF NEEDED

# Load data — prints column list, type/status breakdown, and validation summary
df = pd.read_csv(data_filepath)
# ----------------------------------------------------------------
# Test 1: validate_structure — every source_column in column_config.csv
# should be found in the loaded df.
# ----------------------------------------------------------------
print("TEST 1: validate_structure(df)")
print("=" * 60)
structure_ok = validate_structure(df, config_dir="config/")
print(f"\n>> validate_structure returned: {structure_ok}")

# ----------------------------------------------------------------
# Test 2: validate_lcs_data — Control/LCS subset should have no
# missing values or dtype mismatches in the columns LCSDetector needs.
# ----------------------------------------------------------------
print("\n\nTEST 2: validate_lcs_data(df)")
print("=" * 60)
lcs_validation = validate_lcs_data(df)
print(f"\n>> validate_lcs_data returned:")
lcs_validation 

## 2. Test validate_duplicate_data / validate_replicate_data / validate_srm_data against data/samples batches

Each `data/samples/<PREFIX>_*.csv` file is a small, purpose-built batch for one QC/anomaly
type (`CONTROL_`, `SRMS_`, `DUP_`, `REP_`), with PASS/WARN/FAIL/FAIL_IGN variants.

`validate_*` only checks structural readiness — required columns present, no unexpected
nulls, correct dtypes, no impossible limit values — it does not re-implement pass/fail QC
logic. So every batch below is expected to validate as `status: True`, including the
`*_FAIL_*`/`*_WARN_*` samples: the filename reflects what the detector would conclude about
the measurement, not what the validator checks about the dataframe's shape.

`DUP_*.csv`/`REP_*.csv` are raw per-measurement rows and don't carry the `RPD`/`MEAN_CONC`
columns `validate_duplicate_data`/`validate_replicate_data` expect (those were built against
the pre-paired `notebooks/DUP__historical_reference_model.ipynb` /
`REP__historical_reference_model.ipynb` input shape) — so they're passed through
`add_precision_features` first, reusing the exact derivation from
`notebooks/DUP__current_batch_load.ipynb`.

In [ ]:
from src.data_validator import validate_duplicate_data, validate_replicate_data, validate_srm_data

# ------------------------------------------------------------------
# Helpers for batch-testing validate_* against data/samples/<PREFIX>_*.csv.
# See the markdown note above for why FAIL/WARN samples are still
# expected to validate as status=True.
# ------------------------------------------------------------------

def run_validator_on_batch(pattern: str, validator_fn, prep_fn=None) -> pd.DataFrame:
    """Load every data/samples file matching `pattern`, optionally transform
    it with `prep_fn`, run `validator_fn`, and return one summary row per file."""
    rows = []
    for path in sorted(Path("data/samples").glob(pattern)):
        sample_df = pd.read_csv(path, parse_dates=["ANALYSED_DATE"])
        if prep_fn is not None:
            sample_df = prep_fn(sample_df)
        result = validator_fn(sample_df)
        rows.append({"file": path.name, **result})
    return pd.DataFrame(rows)


def add_precision_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derive MEAN_CONC/ABS_DIFF/RPD from raw paired NUMERIC_FINAL_VALUE /
    PARENT_NUMERIC_FINAL_VALUE rows -- same formula as
    notebooks/DUP__current_batch_load.ipynb, needed because
    validate_duplicate_data / validate_replicate_data expect the
    already-paired reference-model shape, not the raw per-measurement
    data/samples/DUP_*.csv / REP_*.csv rows."""
    out = df.copy()
    out["ABS_DIFF"] = (out["NUMERIC_FINAL_VALUE"] - out["PARENT_NUMERIC_FINAL_VALUE"]).abs()
    out["MEAN_CONC"] = (out["NUMERIC_FINAL_VALUE"] + out["PARENT_NUMERIC_FINAL_VALUE"]) / 2
    out["RPD"] = np.where(
        (out["NUMERIC_FINAL_VALUE"] + out["PARENT_NUMERIC_FINAL_VALUE"]) != 0,
        out["ABS_DIFF"] / out["MEAN_CONC"] * 100,
        np.nan,
    )
    return out

In [ ]:
# ----------------------------------------------------------------
# Test 3: validate_lcs_data across every CONTROL_*.csv sample batch
# (PASS/WARN/FAIL/FAIL_IGN x 2) -- all should validate as structurally OK.
# ----------------------------------------------------------------
print("TEST 3: validate_lcs_data(df) across data/samples/CONTROL_*.csv")
print("=" * 60)
control_summary = run_validator_on_batch("CONTROL_*.csv", validate_lcs_data)
control_summary

In [ ]:
# ----------------------------------------------------------------
# Test 4: validate_srm_data across every SRMS_*.csv sample batch.
# ----------------------------------------------------------------
print("TEST 4: validate_srm_data(df) across data/samples/SRMS_*.csv")
print("=" * 60)
srm_summary = run_validator_on_batch("SRMS_*.csv", validate_srm_data)
srm_summary

In [ ]:
# ----------------------------------------------------------------
# Test 5: validate_duplicate_data across every DUP_*.csv sample batch.
# Raw samples don't carry RPD/MEAN_CONC, so add_precision_features
# derives them first (see notebooks/DUP__current_batch_load.ipynb).
# ----------------------------------------------------------------
print("TEST 5: validate_duplicate_data(df) across data/samples/DUP_*.csv")
print("=" * 60)
dup_summary = run_validator_on_batch("DUP_*.csv", validate_duplicate_data, prep_fn=add_precision_features)
dup_summary

In [ ]:
# ----------------------------------------------------------------
# Test 6: validate_replicate_data across every REP_*.csv sample batch.
# Same prep as duplicates -- REP_*.csv has the identical raw schema,
# just using the non-_DUP limit columns (STAT_DL_VALUE/LIM_REP_VALUE).
# ----------------------------------------------------------------
print("TEST 6: validate_replicate_data(df) across data/samples/REP_*.csv")
print("=" * 60)
rep_summary = run_validator_on_batch("REP_*.csv", validate_replicate_data, prep_fn=add_precision_features)
rep_summary

### Matrix Spike (MS) — Folder 3 validation

The following tests are **additions only**. Tests 1–6 and every existing Control, SRM, Duplicate, and Replicate cell remain unchanged.

**Usage:** Open this notebook from the Folder 3 repository root. Run the Matrix Spike cells beginning at **Test 7**, in order. The older Test 1 uses a Windows-style CSV path and requires the separate QC sample CSV; those original cells have deliberately not been edited. Matrix Spike tests are self-contained and do not depend on Test 1–6 variables.

**Real workbook:** Place `QC Anomaly Training Data v2.xlsx` (or `QC_Anomaly_Training_Data_v2.xlsx`) in Folder 3's `data/raw/`. Raw workbooks are excluded from Git and **are not in the supplied ZIP**. When absent, tests use clearly labelled synthetic data only; synthetic passes do not validate real Datamine results. Set `MS_WORKBOOK_OVERRIDE` in Test 7 to an absolute path if your workbook is stored elsewhere.

In [2]:
# ----------------------------------------------------------------
# TEST 7 — Locate Folder 3, import its actual MS modules, validate input
# ----------------------------------------------------------------
print("TEST 7: Matrix Spike Folder 3 setup and data validation")
print("=" * 60)

import os
import sys
import tempfile
import importlib
from pathlib import Path
import numpy as np
import pandas as pd

# Optional explicit workbook path, if your real training file is elsewhere:
MS_WORKBOOK_OVERRIDE = None  # e.g. r"/Users/.../data/raw/QC Anomaly Training Data v2.xlsx"

# Locate the repository from notebook working directory, not filename "-3".
# This also works when Git has renamed the extracted folder.
ms_roots = [Path.cwd(), *Path.cwd().parents]
MS_ROOT = next((p for p in ms_roots
                if (p / "src/detectors/matrix_spike_detector.py").is_file()
                and (p / "config/matrix_spike_config.yaml").is_file()), None)
if MS_ROOT is None:
    raise FileNotFoundError(
        "Open the Folder 3 repository in VS Code, then run notebook cells from its root. "
        f"Current working directory: {Path.cwd()}"
    )
if str(MS_ROOT) not in sys.path:
    sys.path.insert(0, str(MS_ROOT))

from src.data_validator import validate_matrix_spike_data
from src.detectors.matrix_spike_detector import (
    MSConfig, OUTPUT_COLUMNS, REQUIRED_COLUMNS, load_ms_data,
    load_ms_config, filter_ms_records, run_ms_detection,
    export_results, print_summary, prepare_ms_records,
    _assign_final_risk, _assign_detection_method,
)

MS_CONFIG_PATH = MS_ROOT / "config" / "matrix_spike_config.yaml"
assert MS_CONFIG_PATH.is_file(), MS_CONFIG_PATH
MS_CFG = load_ms_config(str(MS_CONFIG_PATH))  # explicit path bypasses module's incorrect default

# Prefer known source names; avoid accidentally selecting unrelated XLSX files.
ms_raw_dir = MS_ROOT / "data" / "raw"
if MS_WORKBOOK_OVERRIDE:
    MS_WORKBOOK = Path(MS_WORKBOOK_OVERRIDE).expanduser().resolve()
    if not MS_WORKBOOK.is_file():
        raise FileNotFoundError(f"Configured MS workbook not found: {MS_WORKBOOK}")
else:
    choices = [p for p in ms_raw_dir.glob("*.xlsx")
               if "qcanomalytraining" in "".join(c for c in p.stem.casefold() if c.isalnum())]
    choices = sorted(choices)
    if len(choices) > 1:
        raise ValueError(f"Multiple MS training workbooks; specify MS_WORKBOOK_OVERRIDE: {choices}")
    MS_WORKBOOK = choices[0] if choices else None

# Synthetic scenario for repository-only testing; NOT Datamine's real workbook.
def _ms_synthetic_row(i, analyte, value, status, unit="PPM", lot="LOT_A"):
    return {
        "ANALYTICAL_TYPE":"Spike", "QC_TYPE":"MS", "STD_LOT_CODE":lot,
        "STD_CODE":"STD_A", "JOB_CODE":f"JOB_{analyte}_{i:03d}",
        "NUMERIC_FINAL_VALUE":float(value),
        "ANALYSED_DATE":pd.Timestamp("2024-01-01")+pd.Timedelta(days=i),
        "SCHEME_CODE":"SCH", "ANALYTE_CODE":analyte, "UNIT_CODE":unit,
        "SPECIFICATION_CODE":"SPEC_A", "STANDARD_STATUS":status,
        "INTERNAL_TARGET_VALUE":100., "INTERNAL_MIN_VALUE":75.,
        "INTERNAL_MAX_VALUE":125., "INTERNAL_MIN_WARNING_VALUE":85.,
        "INTERNAL_MAX_WARNING_VALUE":115.,
        "INTERNAL_MIN_INCLUSIVE":"Y", "INTERNAL_MAX_INCLUSIVE":"Y",
        "INTERNAL_MIN_WARNING_INCLUSIVE":"Y", "INTERNAL_MAX_WARNING_INCLUSIVE":"Y",
    }

if MS_WORKBOOK is not None:
    MS_REAL_DATA = True
    ms_source = load_ms_data(str(MS_WORKBOOK), sheet_name="SPK(MS) Assessment")
    print("REAL DATAMINE workbook:", MS_WORKBOOK)
else:
    MS_REAL_DATA = False
    synthetic = []
    for i in range(30):
        value = 132. if i >= 15 else 100. + (i % 3)
        synthetic.append(_ms_synthetic_row(i, "AG", value, "UpperFailure" if value > 125 else "Pass"))
        synthetic.append(_ms_synthetic_row(i, "CU", 99. + (i % 4), "Pass"))
    synthetic.append(_ms_synthetic_row(31, "AG", 145, "IgnoredUpperFailure"))
    synthetic.append(_ms_synthetic_row(32, "AG", 100, "Pass", unit="PPM", lot="LOT_A"))
    synthetic.append(_ms_synthetic_row(33, "NA", 100, "Pass"))
    ms_source = pd.DataFrame(synthetic)
    print("SYNTHETIC DATA ONLY: no MS workbook in", ms_raw_dir)
    print("Real-data validation is NOT established. Copy your workbook into Folder 3/data/raw.")

# Folder 3's validate_matrix_spike_data expects INTERNAL lowercase column names,
# while its detector expects RAW UPPERCASE. Adapt only the validator input using
# Folder 3's column_config.csv; do not modify source data or teammate code.
column_map = pd.read_csv(MS_ROOT / "config" / "column_config.csv", encoding="utf-8-sig")
source_to_internal = dict(zip(column_map["source_column"].str.strip().str.upper(),
                              column_map["internal_column"].str.strip()))
ms_validation_frame = ms_source.rename(columns=source_to_internal).copy()
for col in ("analysed_date",):
    if col in ms_validation_frame:
        ms_validation_frame[col] = pd.to_datetime(ms_validation_frame[col], errors="coerce")
for col in ("measured_value", "target_value", "limit_min", "limit_max",
            "limit_min_warning", "limit_max_warning"):
    if col in ms_validation_frame:
        ms_validation_frame[col] = pd.to_numeric(ms_validation_frame[col], errors="coerce")

ms_validator_report = validate_matrix_spike_data(ms_validation_frame)
print("Folder 3:", MS_ROOT)
print("Config:", MS_CONFIG_PATH)
print("Validation:", ms_validator_report)
assert ms_validator_report["status"] is True, f"MS structural readiness failed: {ms_validator_report}"
assert ms_validator_report["n_rows"] == len(ms_source), "MS source subset count differs"
print("PASS: Folder 3 module imports, workbook/schema, and validator readiness.")

TEST 7: Matrix Spike Folder 3 setup and data validation
REAL DATAMINE workbook: /Users/shivanshi/Desktop/QUT-Capstone-Team-32-Datamine-QC-Quality-Control-3/data/raw/QC Anomaly Training Data v2.xlsx
[DataValidator] -- Matrix Spike data validation ----------------------------------------
  Matrix Spike rows (analytical_type=='Spike'): 6,958
    [ok]  analytical_type              expected=str       actual=str
    [ok]  std_lot_code                 expected=str       actual=str
    [ok]  std_code                     expected=str       actual=str
    [ok]  job_code                     expected=str       actual=str
    [ok]  scheme_code                  expected=str       actual=str
    [ok]  analyte_code                 expected=str       actual=str
    [ok]  analysed_date                expected=datetime  actual=datetime64[us]
    [ok]  measured_value               expected=float     actual=float64
    [ok]  target_value                 expected=float     actual=float64
    [ok]  limit_min

In [3]:
# ----------------------------------------------------------------
# TEST 8 — MS-only filtering; 'NA' is not an analyte; no source mutation
# ----------------------------------------------------------------
print("TEST 8: Matrix Spike filtering and NA exclusion")
print("=" * 60)
ms_before = ms_source.copy(deep=True)
ms_filtered = filter_ms_records(ms_source)
pd.testing.assert_frame_equal(ms_source, ms_before)
expected_mask = (
    ms_source["ANALYTICAL_TYPE"].astype("string").str.strip().str.casefold().eq("spike")
    & ms_source["QC_TYPE"].astype("string").str.strip().str.casefold().eq("ms")
    & ms_source["ANALYTE_CODE"].astype("string").str.strip().notna()
    & ms_source["ANALYTE_CODE"].astype("string").str.strip().ne("")
    & ~ms_source["ANALYTE_CODE"].astype("string").str.strip().str.casefold().eq("na")
).fillna(False)
assert len(ms_filtered) == int(expected_mask.sum())
assert ms_filtered["ANALYTE_CODE"].astype(str).str.upper().ne("NA").all()
assert len(ms_filtered) > 0, "No valid MS samples available"
print("Source rows:", len(ms_source), "| Filtered:", len(ms_filtered),
      "| Excluded:", len(ms_source) - len(ms_filtered))
print("PASS: exact filtering, NA exclusion, and source immutability.")

TEST 8: Matrix Spike filtering and NA exclusion
Source rows: 6958 | Filtered: 6872 | Excluded: 86
PASS: exact filtering, NA exclusion, and source immutability.


## 2. test anomaly detection scripts/functions


In [4]:
# ----------------------------------------------------------------
# TEST 9 — Full detector using Folder 3 config (no CLI/default-path dependence)
# ----------------------------------------------------------------
print("TEST 9: Matrix Spike end-to-end detector")
print("=" * 60)
ms_results, ms_drift = run_ms_detection(ms_filtered, MS_CFG)
assert len(ms_results) == len(ms_filtered)
assert not ms_drift.empty
assert len(ms_drift) == ms_filtered.groupby(list(MS_CFG.drift_group_cols), dropna=True).ngroups
print_summary(ms_results, ms_drift)
print("DATA MODE:", "REAL DATAMINE WORKBOOK" if MS_REAL_DATA else "SYNTHETIC SMOKE TEST")
print("PASS: full pipeline and drift groups; zero dropped MS records.")

TEST 9: Matrix Spike end-to-end detector

MS Detection Summary
Total records:     6,872
  Critical        479
  High            200
  Medium         1269
  Low            4551
  Ignored         373

IF anomalies:      366
CCLAS mismatches:  0

Drift series:      86
  Failure       27
  Warning       15
  None          44
DATA MODE: REAL DATAMINE WORKBOOK
PASS: full pipeline and drift groups; zero dropped MS records.


In [5]:
# ----------------------------------------------------------------
# TEST 10 — Output schema and independent target-relative feature calculations
# ----------------------------------------------------------------
print("TEST 10: Matrix Spike output schema and numeric features")
print("=" * 60)
missing = sorted(set(OUTPUT_COLUMNS) - set(ms_results.columns))
print("Missing output columns:", missing)
assert not missing, f"Detector output is missing {missing}"
for col in ["RULE_FLAG", "DRIFT_FLAG", "IF_ANOMALY", "FINAL_RISK", "REASON", "DETECTION_METHOD"]:
    assert ms_results[col].notna().all(), f"{col} has null classifications"
value = pd.to_numeric(ms_results["NUMERIC_FINAL_VALUE"], errors="coerce")
target = pd.to_numeric(ms_results["INTERNAL_TARGET_VALUE"], errors="coerce")
actual = pd.to_numeric(ms_results["TARGET_DEVIATION"], errors="coerce")
assert np.allclose(actual, value-target, rtol=1e-9, atol=1e-9, equal_nan=True)
expected_pct = np.where(target.abs()>1e-9, ((value-target)/target)*100, np.nan)
assert np.allclose(pd.to_numeric(ms_results["TARGET_DEVIATION_PCT"], errors="coerce"),
                   expected_pct, rtol=1e-8, atol=1e-8, equal_nan=True)
upper = pd.to_numeric(ms_results["INTERNAL_MAX_WARNING_VALUE"], errors="coerce") - target
lower = target - pd.to_numeric(ms_results["INTERNAL_MIN_WARNING_VALUE"], errors="coerce")
expected_nd = np.where(value>=target, np.where(abs(upper)>1e-9,(value-target)/upper,np.nan),
                       np.where(abs(lower)>1e-9,(value-target)/lower,np.nan))
assert np.allclose(pd.to_numeric(ms_results["NORM_DEV"], errors="coerce"),
                   expected_nd, rtol=1e-8, atol=1e-8, equal_nan=True)
assert np.allclose(pd.to_numeric(ms_results["ABS_NORM_DEV"], errors="coerce"),
                   np.abs(expected_nd), rtol=1e-8, atol=1e-8, equal_nan=True)
print(f"TARGET_DEVIATION correct: {len(ms_results)}/{len(ms_results)}")
print("PASS: output columns, deviation, percent deviation and normalisation.")

TEST 10: Matrix Spike output schema and numeric features
Missing output columns: []
TARGET_DEVIATION correct: 6872/6872
PASS: output columns, deviation, percent deviation and normalisation.


In [6]:
# ----------------------------------------------------------------
# TEST 11 — Preserve original CCLAS classifications; ignored precedence
# ----------------------------------------------------------------
print("TEST 11: MS CCLAS comparison and ignored precedence")
print("=" * 60)
print(pd.crosstab(ms_results["STANDARD_STATUS"], ms_results["RULE_FLAG"]))
status_to_flag = {
    "UpperFailure":"Failure", "LowerFailure":"Failure",
    "UpperWarning":"Warning", "LowerWarning":"Warning",
    "Pass":"Pass", "IgnoredUpperFailure":"Ignored", "IgnoredLowerFailure":"Ignored",
}
recognized = ms_results["STANDARD_STATUS"].isin(status_to_flag)
if recognized.any():
    exp = ms_results.loc[recognized,"STANDARD_STATUS"].map(status_to_flag)
    assert (ms_results.loc[recognized,"RULE_FLAG"] == exp).all(), "CCLAS / rule flags disagree"
    assert not ms_results.loc[recognized,"CCLAS_MISMATCH"].astype(bool).any()
else:
    print("WARNING: No recognized CCLAS statuses to compare; review source statuses.")
ignored = ms_results["STANDARD_STATUS"].isin(["IgnoredUpperFailure", "IgnoredLowerFailure"])
assert (ms_results.loc[ignored,"RULE_FLAG"]=="Ignored").all()
assert (ms_results.loc[ignored,"FINAL_RISK"]=="Ignored").all()
assert (~ms_results.loc[ignored,"IF_ANOMALY"].astype(bool)).all()
assert (ms_results.loc[ms_results["RULE_FLAG"]=="Failure","FINAL_RISK"]=="Critical").all()
print("Recognized CCLAS statuses:", int(recognized.sum()), "| mismatches:",
      int(ms_results["CCLAS_MISMATCH"].sum()), "| ignored:", int(ignored.sum()))
print("PASS: rule-based agreement and ignored-record precedence.")

TEST 11: MS CCLAS comparison and ignored precedence
RULE_FLAG            Failure  Ignored  Pass  Warning
STANDARD_STATUS                                     
IgnoredLowerFailure        0      152     0        0
IgnoredUpperFailure        0      221     0        0
LowerFailure             249        0     0        0
LowerWarning               0        0     0      589
Pass                       0        0  4684        0
UpperFailure             230        0     0        0
UpperWarning               0        0     0      747
Recognized CCLAS statuses: 6872 | mismatches: 0 | ignored: 373
PASS: rule-based agreement and ignored-record precedence.


In [7]:
# ----------------------------------------------------------------
# TEST 12 — Verify exact drift confirmation rows; no later propagation
# ----------------------------------------------------------------
print("TEST 12: Matrix Spike historical drift and confirmation-only flags")
print("=" * 60)
assert set(ms_drift["drift_level"]).issubset({"None", "Warning", "Failure"})
assert ms_drift.groupby(list(MS_CFG.drift_group_cols), dropna=False).size().eq(1).all()

# Recreate only the detector's documented sort order to attach the existing
# UNIT_CODE back to outputs, since Folder 3's OUTPUT_COLUMNS currently omits it.
# Verify source/result identity before using this visualiser-only adapter.
ms_ordered = prepare_ms_records(ms_filtered.copy(), MS_CFG)
ms_ordered = ms_ordered.sort_values(list(MS_CFG.drift_group_cols)+["ANALYSED_DATE"],
                                   kind="stable").reset_index(drop=True)
assert len(ms_ordered)==len(ms_results)
for col in ("STD_LOT_CODE", "SCHEME_CODE", "ANALYTE_CODE", "JOB_CODE", "STANDARD_STATUS"):
    assert ms_ordered[col].astype(str).tolist()==ms_results[col].astype(str).tolist(), (
        f"Cannot reconstruct UNIT_CODE safely; row order mismatch on {col}"
    )
assert (ms_ordered["ANALYSED_DATE"].to_numpy() == ms_results["ANALYSED_DATE"].to_numpy()).all()
assert np.allclose(ms_ordered["NUMERIC_FINAL_VALUE"].to_numpy(dtype=float),
                   ms_results["NUMERIC_FINAL_VALUE"].to_numpy(dtype=float), equal_nan=True)
ms_plot_ready = ms_results.copy()
ms_plot_ready["UNIT_CODE"] = ms_ordered["UNIT_CODE"].to_numpy()

expected_flag = pd.Series("None", index=ms_plot_ready.index, dtype=object)
expected_direction = pd.Series("", index=ms_plot_ready.index, dtype=object)
expected_onset = pd.Series(pd.NaT, index=ms_plot_ready.index, dtype="datetime64[ns]")
for _, d in ms_drift.iterrows():
    match = pd.Series(True,index=ms_plot_ready.index)
    for key in MS_CFG.drift_group_cols:
        match &= ms_plot_ready[key].eq(d[key])
    group = ms_plot_ready.loc[match].sort_values("ANALYSED_DATE")
    for level, prefix in [("Warning","warning"),("Failure","failure")]:
        if not bool(d[f"{prefix}_drift_detected"]):
            continue
        onset = pd.to_datetime(d[f"{prefix}_drift_start"],errors="coerce")
        assert pd.notna(onset), "Detected drift must have an onset"
        idx = group.index[group["ANALYSED_DATE"]>=onset][:MS_CFG.drift_consecutive_required]
        expected_flag.loc[idx]=level  # Failure overrides Warning if they overlap
        expected_direction.loc[idx]=d[f"{prefix}_direction"] or ""
        expected_onset.loc[idx]=onset
assert expected_flag.eq(ms_plot_ready["DRIFT_FLAG"]).all(), "Exact row-level drift flags are incorrect"
assert expected_direction.eq(ms_plot_ready["DRIFT_DIRECTION"]).all(), "Drift direction mismatch"
assert pd.to_datetime(ms_plot_ready["DRIFT_ONSET"]).equals(pd.to_datetime(expected_onset)), "Drift onset mismatch"
print("Historical series:", ms_drift["drift_level"].value_counts().to_dict())
print("Row-level flags:", ms_results["DRIFT_FLAG"].value_counts().to_dict())
print("PASS: exact first confirming rows, direction and onset; no blanket propagation.")

TEST 12: Matrix Spike historical drift and confirmation-only flags
Historical series: {'None': 44, 'Failure': 27, 'Warning': 15}
Row-level flags: {'None': 6696, 'Warning': 95, 'Failure': 81}
PASS: exact first confirming rows, direction and onset; no blanket propagation.


In [8]:
# ----------------------------------------------------------------
# TEST 13 — Isolation Forest, final risk and traceable detection methods
# ----------------------------------------------------------------
print("TEST 13: Matrix Spike ML, final risk and detection method")
print("=" * 60)
assert ms_results["IF_ANOMALY"].dtype==bool or ms_results["IF_ANOMALY"].isin([True,False]).all()
assert ms_results["IF_MODEL"].astype(str).str.len().gt(0).all()
assert ms_results["FINAL_RISK"].isin(["Low","Medium","High","Critical","Ignored"]).all()
for _, row in ms_results.iterrows():
    ignored = row["STANDARD_STATUS"] in MS_CFG.ignored_statuses
    risk = _assign_final_risk(row["RULE_FLAG"], bool(row["IF_ANOMALY"]),row["DRIFT_FLAG"],ignored)
    method = _assign_detection_method(row["RULE_FLAG"], bool(row["IF_ANOMALY"]),row["DRIFT_FLAG"],ignored)
    assert row["FINAL_RISK"]==risk
    assert row["DETECTION_METHOD"]==method

# Check expected combinations independently of the dataset distribution.
for rule, if_flag, drift_flag, ignored, expected_risk, expected_method in [
    ("Failure",False,"None",False,"Critical","Rule-based"),
    ("Warning",False,"None",False,"Medium","Rule-based"),
    ("Warning",True,"None",False,"High","Rule-based + IsolationForest"),
    ("Pass",True,"None",False,"Medium","IsolationForest"),
    ("Pass",False,"Failure",False,"High","Drift"),
    ("Pass",False,"Warning",False,"Medium","Drift"),
    ("Pass",False,"None",False,"Low","None"),
    ("Failure",True,"Failure",True,"Ignored","Ignored"),
]:
    assert _assign_final_risk(rule,if_flag,drift_flag,ignored)==expected_risk
    assert _assign_detection_method(rule,if_flag,drift_flag,ignored)==expected_method
print("IF anomalies:",int(ms_results["IF_ANOMALY"].sum()))
print("IF anomalies on CCLAS Pass:",int((ms_results["IF_ANOMALY"] & ms_results["STANDARD_STATUS"].eq("Pass")).sum()))
print("Risk counts:",ms_results["FINAL_RISK"].value_counts().to_dict())
print("PASS: model/ML flags and 8 explicit risk and method cases.")

TEST 13: Matrix Spike ML, final risk and detection method
IF anomalies: 366
IF anomalies on CCLAS Pass: 69
Risk counts: {'Low': 4551, 'Medium': 1269, 'Critical': 479, 'Ignored': 373, 'High': 200}
PASS: model/ML flags and 8 explicit risk and method cases.


## 3. test anomaly visualisers scripts/functions


### Matrix Spike visualisation integration (Folder 3 actual filenames)

The real visualiser is `src/visualisers/matrix_spike_visualize.py`. The detector currently omits `UNIT_CODE` from exported results although the per-series chart generator requires it. Test 14 runs the original plotting functions using a **test-only, identity-checked enriched copy** and reports the production interface gap; it does not modify the detector, saved detector output or other teammates' code.

In [9]:
# ----------------------------------------------------------------
# TEST 14 — Actual visualiser API / actual PNGs (not merely syntax)
# ----------------------------------------------------------------
print("TEST 14: Folder 3 Matrix Spike visualisations")
print("=" * 60)
from src.visualisers.matrix_spike_visualize import (
    generate_overall_detection_plots, generate_ms_plots,
)
MS_TEST_DIR = Path(tempfile.mkdtemp(prefix="datamine_ms_workflow_"))
overall_plots = generate_overall_detection_plots(ms_results, ms_drift, MS_TEST_DIR / "overall")
assert len(overall_plots)>=9
assert all(p.exists() and p.stat().st_size>0 for p in overall_plots.values())

# Verify each exact expected series has chart from unmodified plotting function.
per_series = generate_ms_plots(ms_plot_ready, ms_drift, MS_TEST_DIR / "drift", only_flagged=True,
                              rolling_window=MS_CFG.drift_rolling_window)
expected_keys = {tuple(str(d[k]) for k in MS_CFG.drift_group_cols)
                 for _,d in ms_drift.loc[ms_drift["drift_level"]!="None"].iterrows()}
assert set(per_series)==expected_keys, "Some drift series are missing plots"
assert all(p.exists() and p.stat().st_size>0 for p in per_series.values())
print("Overall chart files:",len(overall_plots), "| Drift history chart files:",len(per_series))
print("Temporary chart location:",MS_TEST_DIR)
if "UNIT_CODE" not in ms_results.columns:
    print("INTEGRATION GAP: detector's saved CSV omits UNIT_CODE, so running the visualiser")
    print("standalone against exported CSV will fail for per-series charts. Test used")
    print("a verified source-only UNIT_CODE adapter; detector code was NOT modified.")
print("PASS: actual MS plotting functions produced PNG files.")

TEST 14: Folder 3 Matrix Spike visualisations


Matplotlib is building the font cache; this may take a moment.


Overall chart files: 10 | Drift history chart files: 42
Temporary chart location: /var/folders/50/8f_xc_fn2fn8ttr0vjcw56gc0000gn/T/datamine_ms_workflow_8e8als2a
INTEGRATION GAP: detector's saved CSV omits UNIT_CODE, so running the visualiser
standalone against exported CSV will fail for per-series charts. Test used
a verified source-only UNIT_CODE adapter; detector code was NOT modified.
PASS: actual MS plotting functions produced PNG files.


## 4. test report creation

### Matrix Spike report/export validation

Test 15 writes outputs only into the operating system's temporary directory. It checks the same CSV filenames and row/series counts consumed by Folder 3's report visualiser, without altering any shared report or other QC output.

In [ ]:
# ----------------------------------------------------------------
# TEST 15 — Exact detector CSV export and visualiser CSV reload contract
# ----------------------------------------------------------------
print("TEST 15: MS CSV export and report integration")
print("=" * 60)
from src.visualisers.matrix_spike_visualize import load_detector_outputs
export_paths=export_results(ms_results,ms_drift,MS_TEST_DIR/"exports")
expected_filenames={
    "full":"ms_detection_results.csv",
    "flagged":"ms_detection_flagged.csv",
    "drift":"ms_drift_results.csv",
    "drift_flagged":"ms_drift_flagged.csv",
}
for tag,filename in expected_filenames.items():
    assert tag in export_paths, f"Missing MS export: {tag}"
    assert export_paths[tag].name==filename
    assert export_paths[tag].is_file() and export_paths[tag].stat().st_size>0
reloaded,drift_reloaded=load_detector_outputs(export_paths["full"],export_paths["drift"])
assert len(reloaded)==len(ms_results)
assert len(drift_reloaded)==len(ms_drift)
flagged_csv=pd.read_csv(export_paths["flagged"],keep_default_na=False)
assert len(flagged_csv)==int(ms_results["FINAL_RISK"].isin(["Critical","High","Medium"]).sum())
flagged_series_csv=pd.read_csv(export_paths["drift_flagged"],keep_default_na=False)
assert len(flagged_series_csv)==int(ms_drift["drift_level"].ne("None").sum())
assert "FINAL_RISK" in reloaded and "drift_level" in drift_reloaded
print("CSV rows:",len(reloaded),"| flagged:",len(flagged_csv),"| historical groups:",len(drift_reloaded))
print("CSV files:",*[p.name for p in export_paths.values()])
print("DATA MODE:","REAL DATAMINE WORKBOOK" if MS_REAL_DATA else "SYNTHETIC SMOKE TEST")
if not MS_REAL_DATA:
    print("REMINDER: Real workbook validation pending; workbook is not shipped in Git ZIP.")
print("PASS: real exporter / reloader / report filenames and record counts.")